In [3]:
import os, random, argparse, itertools, numpy as np, scipy.io as sio
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import scale
from _utils import MMDataset, clustering_acc, overall_performance_report

class DUCMME(nn.Module):
    def __init__(self, embed_dim=200, num_samples=10000, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512], n_clusters=10, alpha=1.0):
        super(DUCMME, self).__init__()
        self.embed_dim = embed_dim; self.num_samples = num_samples; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims; self.n_clusters = n_clusters; self.alpha = alpha
        # 1. Multi-view Feature Extraction by Fusion-Net
        self.fusion_net_encoder = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                               nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)]) # encode each view
        self.fusion_net_mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=10, batch_first=True) # batch_first=True: (batch_size, seq_len, hidden_dim)
        self.fusion_net_linear = nn.Linear(3*embed_dim, embed_dim) # linear projection of the fused encoded features
        # 2. Uncertainty-Aware Reconstruction by Reconstruction-Net and Uncertainty-Net
        self.reconstruct_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # reconstruct each view
        self.uncertainty_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # predict uncertainty for each view
        # 3. Deep Embedding Clustering by DEC
        self._cluster_centers = nn.Parameter(torch.Tensor(self.n_clusters, self.embed_dim))
        nn.init.xavier_uniform_(self._cluster_centers.data)
        
    def forward_embedding(self, x):
        if self.training:
            x = [x[i] + 0.1 * torch.randn_like(x[i]) for i in range(self.num_views)] # add noise to the input data
        encoded_output_list = [self.fusion_net_encoder[i](x[i]) for i in range(self.num_views)] # encode each view
        encoded_output_list = torch.stack(encoded_output_list, dim=1) # stack the encoded features from all views, (batch_size, num_views, embed_dim)
        encoded_output_list, _ = self.fusion_net_mha(encoded_output_list, encoded_output_list, encoded_output_list) # fuse the encoded features from all views by a multihead attention, (batch_size, num_views, embed_dim)
        encoded_output_list = encoded_output_list.contiguous().view(encoded_output_list.shape[0], -1) # flatten the encoded features, (batch_size, num_views*embed_dim)
        embedding = self.fusion_net_linear(encoded_output_list) # linear projection of the fused encoded features
        return embedding # get the embedding of the latent space H, (batch_size, embed_dim)

    def forward_uncertainty_aware_reconstruction(self, x):
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        reconstructions = [self.reconstruct_net_list[i](embedding) for i in range(self.num_views)] # reconstruct each view
        uncertainties = [self.uncertainty_net_list[i](embedding) for i in range(self.num_views)] # predict uncertainty for each view
        return reconstructions, uncertainties
        
    def forward_similarity_matrix_q(self, x): # calculate the similarity matrix q using t-distribution
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        q = 1.0 / (1.0 + torch.sum((embedding.unsqueeze(1) - self._cluster_centers) ** 2, dim=2) / self.alpha) # shape: [batch_size, n_clusters]
        q = q ** ((self.alpha + 1.0) / 2.0) # , shape: [batch_size, n_clusters]
        q = q / torch.sum(q, dim=1, keepdim=True) # Normalize q to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return q, embedding # q can be regarded as the probability of the sample belonging to each cluster
    
    @property
    def cluster_centers(self):
        return self._cluster_centers.data.detach().cpu().numpy() # shape: (n_clusters, embed_dim)
    
    @cluster_centers.setter
    def cluster_centers(self, centers): # shape: (n_clusters, embed_dim)
        centers = torch.tensor(centers, dtype=torch.float32, device=self._cluster_centers.device)
        self._cluster_centers.data.copy_(centers) # copy the cluster centers to the model, set the cluster centers to the new cluster centers
        
    @staticmethod
    def target_distribution(q):
        weight = q ** 2 / torch.sum(q, dim=0) # shape: [batch_size, n_clusters]
        p = weight / torch.sum(weight, dim=1, keepdim=True) # Normalize p to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return p.clone().detach()
    
    def reconstruction_loss(self, x):
        # x = [x[i] + 0.1 * torch.randn_like(x[i]) for i in range(self.num_views)] # add noise to the input data
        x_rec, _ = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        return sum([F.mse_loss(x_rec[v], x[v], reduction='mean') for v in range(self.num_views)]) # sum the losses from all views
    
    def uncertainty_aware_reconstruction_loss(self, x):
        # x = [x[i] + 0.1 * torch.randn_like(x[i]) for i in range(self.num_views)] # add noise to the input data
        x_rec, log_sigma_2 = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        # Clip log_sigma_2 to prevent numerical instability (exp(-log_sigma_2) overflow/underflow)
        # Clamp to reasonable range: -10 to 10, which corresponds to sigma^2 from exp(-10) to exp(10)
        log_sigma_2 = [torch.clamp(log_sigma_2[v], min=-10.0, max=10.0) for v in range(self.num_views)] # shape: [num_views, batch_size, feature_dim] for numerically stable computation
        return sum([0.5 * torch.mean((x_rec[v] - x[v])**2 * torch.exp(-log_sigma_2[v]) + log_sigma_2[v]) for v in range(self.num_views)]) # uncertainty is equal to log_sigma_2
    
    def clustering_loss(self, x, p):
        # x = [x[i] + 0.1 * torch.randn_like(x[i]) for i in range(self.num_views)] # add noise to the input data
        q, _ = self.forward_similarity_matrix_q(x) # shape: [batch_size, n_clusters]
        return F.kl_div(q.log(), p, reduction='batchmean') # shape: ()

In [4]:
random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0) # Set random seed for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = MMDataset('./data/data_sc_multiomics/NEAT/', concat_data=False)
data = [x.clone().to(device) for x in dataset.X]; label = dataset.Y.clone().numpy()
data_views = dataset.data_views; data_samples = dataset.data_samples; data_features = dataset.data_features; data_categories = dataset.categories
label_dict = dataset.get_label_to_name()

dataloader = torch.utils.data.DataLoader(dataset, batch_size=200, shuffle=True)
model = DUCMME(embed_dim=20, feature_dims=data_features, num_views=data_views, hidden_dims=[512, 512, 512], num_samples=data_samples, n_clusters=data_categories, alpha=1.0).to(device)
## === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
print("\n=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===")
print("Basic reconstruction training...")
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
losses_convergence_plot_reconstruction = []
for epoch in range(200):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Pretraining completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
print("Uncertainty-aware reconstruction finetuning...")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses_convergence_plot_reconstruction_finetuning = []
for epoch in range(100):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.uncertainty_aware_reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction_finetuning.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Finetuning completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")

## === Stage 2: Deep Embedding Clustering by DEC ===
print("\n=== Stage 2: Deep Embedding Clustering ===")
print("Cluster center initialization...")
model.eval()
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Cluster center initialization completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
model.cluster_centers = kmeans.cluster_centers_ # shape: (n_clusters, embed_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)
losses = []
acc_convergence_plot = []
nmi_convergence_plot = []
asw_convergence_plot = []
ari_convergence_plot = []
embeddings_dict = {}
preds_dict = {}
for epoch in range(200):
    # Update target distribution periodically
    if epoch % 1 == 0:
        model.eval()
        with torch.no_grad():
            q, embedding = model.forward_similarity_matrix_q(data)
            p = model.target_distribution(q)
        y_pred = torch.argmax(q, dim=1).cpu().numpy()
        acc_val = clustering_acc(label, y_pred)
        nmi_val = normalized_mutual_info_score(label, y_pred)
        asw_val = silhouette_score(embedding.cpu().numpy(), y_pred) if np.unique(y_pred).shape[0] > 1 else 0.0
        ari_val = adjusted_rand_score(label, y_pred)
        acc_convergence_plot.append(acc_val)
        nmi_convergence_plot.append(nmi_val)
        asw_convergence_plot.append(asw_val)
        ari_convergence_plot.append(ari_val)
        embeddings_dict[epoch] = embedding
        preds_dict[epoch] = y_pred
        if epoch == 0:
            delta_label = 1.0
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
        else:
            delta_label = np.sum(y_pred != y_pred_last).astype(np.float32) / y_pred.shape[0]
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
            if delta_label < 0.001:
                convergence_epoch = epoch
                print('Converged, stopping training...')
                # break
    # Training based on the target distribution that is updated periodically
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.clustering_loss(x, p[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
print(f'Final ACC: {clustering_acc(label, y_pred):.4f}')

modality_rna shape: (8472, 100)
modality_protein shape: (8472, 7)
modality_atac shape: (8472, 100)

=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
Basic reconstruction training...
Epoch 0 completed. Average Loss: 0.2093
Epoch 1 completed. Average Loss: 0.0297
Epoch 2 completed. Average Loss: 0.0233
Epoch 3 completed. Average Loss: 0.0236
Epoch 4 completed. Average Loss: 0.0227
Epoch 5 completed. Average Loss: 0.0230
Epoch 6 completed. Average Loss: 0.0231
Epoch 7 completed. Average Loss: 0.0226
Epoch 8 completed. Average Loss: 0.0225
Epoch 9 completed. Average Loss: 0.0217
Epoch 10 completed. Average Loss: 0.0223
Epoch 11 completed. Average Loss: 0.0224
Epoch 12 completed. Average Loss: 0.0229
Epoch 13 completed. Average Loss: 0.0218
Epoch 14 completed. Average Loss: 0.0221
Epoch 15 completed. Average Loss: 0.0214
Epoch 16 completed. Average Loss: 0.0213
Epoch 17 completed. Average Loss: 0.0213
Epoch 18 completed. Average Loss: 0.0214
Epoch 19 completed. Average Loss: 0.

In [3]:
# modality_rna shape: (8472, 100)
# modality_protein shape: (8472, 7)
# modality_atac shape: (8472, 100)
# === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
# Basic reconstruction training...
# Pretraining completed. ACC: 0.4366, NMI: 0.1882, ASW: 0.0679, ARI: 0.1501
# Uncertainty-aware reconstruction finetuning...
# Finetuning completed. ACC: 0.4225, NMI: 0.1516, ASW: 0.0815, ARI: 0.1270
# === Stage 2: Deep Embedding Clustering ===
# Cluster center initialization...
# Cluster center initialization completed. ACC: 0.4749, NMI: 0.2810, ASW: 0.0709, ARI: 0.2088
# [Epoch 0] ACC: 0.4749, NMI: 0.2810, ASW: 0.0709, ARI: 0.2088, Delta: 1.0000
# Epoch 0 completed. Average Loss: 0.1158
# [Epoch 1] ACC: 0.4698, NMI: 0.2915, ASW: 0.1034, ARI: 0.2044, Delta: 0.1369
# Epoch 1 completed. Average Loss: 0.1076
# [Epoch 2] ACC: 0.4304, NMI: 0.2806, ASW: 0.1165, ARI: 0.1737, Delta: 0.0984
# Epoch 2 completed. Average Loss: 0.1210
# [Epoch 3] ACC: 0.4432, NMI: 0.2863, ASW: 0.1290, ARI: 0.1725, Delta: 0.1023
# Epoch 3 completed. Average Loss: 0.1236
# [Epoch 4] ACC: 0.4445, NMI: 0.2882, ASW: 0.1408, ARI: 0.1803, Delta: 0.0530
# Epoch 4 completed. Average Loss: 0.1369
# [Epoch 5] ACC: 0.4337, NMI: 0.2690, ASW: 0.1370, ARI: 0.1699, Delta: 0.1288
# Epoch 5 completed. Average Loss: 0.1240
# [Epoch 6] ACC: 0.4322, NMI: 0.2619, ASW: 0.1370, ARI: 0.1603, Delta: 0.0817
# Epoch 6 completed. Average Loss: 0.1150
# [Epoch 7] ACC: 0.4337, NMI: 0.2649, ASW: 0.1403, ARI: 0.1657, Delta: 0.0641
# Epoch 7 completed. Average Loss: 0.1146
# [Epoch 8] ACC: 0.4175, NMI: 0.2584, ASW: 0.1475, ARI: 0.1623, Delta: 0.0561
# Epoch 8 completed. Average Loss: 0.1153
# [Epoch 9] ACC: 0.4252, NMI: 0.2607, ASW: 0.1489, ARI: 0.1623, Delta: 0.0631
# Epoch 9 completed. Average Loss: 0.1083
# [Epoch 10] ACC: 0.4359, NMI: 0.2644, ASW: 0.1481, ARI: 0.1687, Delta: 0.0648
# Epoch 10 completed. Average Loss: 0.0984
# [Epoch 11] ACC: 0.4299, NMI: 0.2659, ASW: 0.1534, ARI: 0.1703, Delta: 0.0596
# Epoch 11 completed. Average Loss: 0.0997
# [Epoch 12] ACC: 0.4351, NMI: 0.2672, ASW: 0.1537, ARI: 0.1761, Delta: 0.0347
# Epoch 12 completed. Average Loss: 0.1015
# [Epoch 13] ACC: 0.4295, NMI: 0.2682, ASW: 0.1601, ARI: 0.1747, Delta: 0.0345
# Epoch 13 completed. Average Loss: 0.1041
# [Epoch 14] ACC: 0.4413, NMI: 0.2720, ASW: 0.1604, ARI: 0.1832, Delta: 0.0654
# Epoch 14 completed. Average Loss: 0.1049
# [Epoch 15] ACC: 0.4308, NMI: 0.2695, ASW: 0.1655, ARI: 0.1758, Delta: 0.0545
# Epoch 15 completed. Average Loss: 0.0964
# [Epoch 16] ACC: 0.4280, NMI: 0.2669, ASW: 0.1665, ARI: 0.1724, Delta: 0.0364
# Epoch 16 completed. Average Loss: 0.1005
# [Epoch 17] ACC: 0.4233, NMI: 0.2600, ASW: 0.1722, ARI: 0.1567, Delta: 0.0731
# Epoch 17 completed. Average Loss: 0.0935
# [Epoch 18] ACC: 0.4155, NMI: 0.2582, ASW: 0.1719, ARI: 0.1475, Delta: 0.0322
# Epoch 18 completed. Average Loss: 0.0920
# [Epoch 19] ACC: 0.4234, NMI: 0.2627, ASW: 0.1722, ARI: 0.1616, Delta: 0.0492
# Epoch 19 completed. Average Loss: 0.0812
# [Epoch 20] ACC: 0.4273, NMI: 0.2639, ASW: 0.1698, ARI: 0.1639, Delta: 0.0224
# Epoch 20 completed. Average Loss: 0.0745
# [Epoch 21] ACC: 0.4306, NMI: 0.2630, ASW: 0.1698, ARI: 0.1577, Delta: 0.0356
# Epoch 21 completed. Average Loss: 0.0702
# [Epoch 22] ACC: 0.4319, NMI: 0.2632, ASW: 0.1651, ARI: 0.1631, Delta: 0.0378
# Epoch 22 completed. Average Loss: 0.0740
# [Epoch 23] ACC: 0.4237, NMI: 0.2605, ASW: 0.1679, ARI: 0.1513, Delta: 0.0476
# Epoch 23 completed. Average Loss: 0.0667
# [Epoch 24] ACC: 0.4236, NMI: 0.2584, ASW: 0.1650, ARI: 0.1487, Delta: 0.0378
# Epoch 24 completed. Average Loss: 0.0721
# [Epoch 25] ACC: 0.4195, NMI: 0.2538, ASW: 0.1640, ARI: 0.1456, Delta: 0.0434
# Epoch 25 completed. Average Loss: 0.0641
# [Epoch 26] ACC: 0.4083, NMI: 0.2538, ASW: 0.1605, ARI: 0.1432, Delta: 0.0608
# Epoch 26 completed. Average Loss: 0.0661
# [Epoch 27] ACC: 0.4038, NMI: 0.2520, ASW: 0.1611, ARI: 0.1363, Delta: 0.0341
# Epoch 27 completed. Average Loss: 0.0682
# [Epoch 28] ACC: 0.3983, NMI: 0.2476, ASW: 0.1613, ARI: 0.1316, Delta: 0.0310
# Epoch 28 completed. Average Loss: 0.0662
# [Epoch 29] ACC: 0.3857, NMI: 0.2430, ASW: 0.1599, ARI: 0.1278, Delta: 0.1104
# Epoch 29 completed. Average Loss: 0.0719
# [Epoch 30] ACC: 0.3981, NMI: 0.2338, ASW: 0.1629, ARI: 0.1176, Delta: 0.0656
# Epoch 30 completed. Average Loss: 0.0700
# [Epoch 31] ACC: 0.4128, NMI: 0.2321, ASW: 0.1670, ARI: 0.1201, Delta: 0.0752
# Epoch 31 completed. Average Loss: 0.0700
# [Epoch 32] ACC: 0.4391, NMI: 0.2390, ASW: 0.1772, ARI: 0.1377, Delta: 0.1238
# Epoch 32 completed. Average Loss: 0.0703
# [Epoch 33] ACC: 0.4461, NMI: 0.2403, ASW: 0.1742, ARI: 0.1409, Delta: 0.0513
# Epoch 33 completed. Average Loss: 0.0568
# [Epoch 34] ACC: 0.4474, NMI: 0.2374, ASW: 0.1721, ARI: 0.1374, Delta: 0.0741
# Epoch 34 completed. Average Loss: 0.0614
# [Epoch 35] ACC: 0.4609, NMI: 0.2456, ASW: 0.1716, ARI: 0.1531, Delta: 0.0456
# Epoch 35 completed. Average Loss: 0.0628
# [Epoch 36] ACC: 0.4588, NMI: 0.2462, ASW: 0.1719, ARI: 0.1485, Delta: 0.0495
# Epoch 36 completed. Average Loss: 0.0683
# [Epoch 37] ACC: 0.4451, NMI: 0.2408, ASW: 0.1633, ARI: 0.1293, Delta: 0.0784
# Epoch 37 completed. Average Loss: 0.0576
# [Epoch 38] ACC: 0.4500, NMI: 0.2407, ASW: 0.1632, ARI: 0.1329, Delta: 0.0326
# Epoch 38 completed. Average Loss: 0.0602
# [Epoch 39] ACC: 0.4657, NMI: 0.2470, ASW: 0.1700, ARI: 0.1496, Delta: 0.0621
# Epoch 39 completed. Average Loss: 0.0598
# [Epoch 40] ACC: 0.4777, NMI: 0.2498, ASW: 0.1722, ARI: 0.1587, Delta: 0.0413
# Epoch 40 completed. Average Loss: 0.0651
# [Epoch 41] ACC: 0.4852, NMI: 0.2551, ASW: 0.1715, ARI: 0.1685, Delta: 0.0368
# Epoch 41 completed. Average Loss: 0.0575
# [Epoch 42] ACC: 0.4835, NMI: 0.2561, ASW: 0.1717, ARI: 0.1654, Delta: 0.0322
# Epoch 42 completed. Average Loss: 0.0544
# [Epoch 43] ACC: 0.5011, NMI: 0.2659, ASW: 0.1755, ARI: 0.1807, Delta: 0.0425 # select this result
# Epoch 43 completed. Average Loss: 0.0607
# [Epoch 44] ACC: 0.5138, NMI: 0.2770, ASW: 0.1871, ARI: 0.1964, Delta: 0.0476
# Epoch 44 completed. Average Loss: 0.0606
# [Epoch 45] ACC: 0.5142, NMI: 0.2754, ASW: 0.1733, ARI: 0.1933, Delta: 0.0375
# Epoch 45 completed. Average Loss: 0.0596
# [Epoch 46] ACC: 0.5194, NMI: 0.2763, ASW: 0.1591, ARI: 0.2022, Delta: 0.0781
# Epoch 46 completed. Average Loss: 0.0610
# [Epoch 47] ACC: 0.5299, NMI: 0.2835, ASW: 0.1615, ARI: 0.2139, Delta: 0.0620
# Epoch 47 completed. Average Loss: 0.0615
# [Epoch 48] ACC: 0.5366, NMI: 0.2846, ASW: 0.1558, ARI: 0.2214, Delta: 0.0608
# Epoch 48 completed. Average Loss: 0.0556
# [Epoch 49] ACC: 0.5341, NMI: 0.2885, ASW: 0.1757, ARI: 0.2142, Delta: 0.1014
# Epoch 49 completed. Average Loss: 0.0573
# [Epoch 50] ACC: 0.5316, NMI: 0.2809, ASW: 0.1635, ARI: 0.2097, Delta: 0.0563
# Epoch 50 completed. Average Loss: 0.0569
# [Epoch 51] ACC: 0.5356, NMI: 0.2953, ASW: 0.1946, ARI: 0.2099, Delta: 0.1034
# Epoch 51 completed. Average Loss: 0.0618
# [Epoch 52] ACC: 0.5464, NMI: 0.2972, ASW: 0.1715, ARI: 0.2290, Delta: 0.0724
# Epoch 52 completed. Average Loss: 0.0624
# [Epoch 53] ACC: 0.5268, NMI: 0.2960, ASW: 0.1991, ARI: 0.1972, Delta: 0.1170
# Epoch 53 completed. Average Loss: 0.0631
# [Epoch 54] ACC: 0.5407, NMI: 0.3005, ASW: 0.2020, ARI: 0.2171, Delta: 0.0678
# Epoch 54 completed. Average Loss: 0.0644
# [Epoch 55] ACC: 0.5465, NMI: 0.3045, ASW: 0.1884, ARI: 0.2254, Delta: 0.0289
# Epoch 55 completed. Average Loss: 0.0675
# [Epoch 56] ACC: 0.5407, NMI: 0.3021, ASW: 0.1937, ARI: 0.2189, Delta: 0.0374
# Epoch 56 completed. Average Loss: 0.0633
# [Epoch 57] ACC: 0.5346, NMI: 0.3022, ASW: 0.1974, ARI: 0.2103, Delta: 0.0384
# Epoch 57 completed. Average Loss: 0.0619
# [Epoch 58] ACC: 0.5336, NMI: 0.3023, ASW: 0.2043, ARI: 0.2059, Delta: 0.0329
# Epoch 58 completed. Average Loss: 0.0564
# [Epoch 59] ACC: 0.5315, NMI: 0.3003, ASW: 0.2074, ARI: 0.2047, Delta: 0.0229
# Epoch 59 completed. Average Loss: 0.0599
# [Epoch 60] ACC: 0.5279, NMI: 0.2966, ASW: 0.2139, ARI: 0.2022, Delta: 0.0358
# Epoch 60 completed. Average Loss: 0.0631
# [Epoch 61] ACC: 0.5401, NMI: 0.3028, ASW: 0.1976, ARI: 0.2178, Delta: 0.0436
# Epoch 61 completed. Average Loss: 0.0606
# [Epoch 62] ACC: 0.5333, NMI: 0.2983, ASW: 0.1905, ARI: 0.2086, Delta: 0.0323
# Epoch 62 completed. Average Loss: 0.0681
# [Epoch 63] ACC: 0.5362, NMI: 0.3019, ASW: 0.1793, ARI: 0.2093, Delta: 0.0421
# Epoch 63 completed. Average Loss: 0.0639
# [Epoch 64] ACC: 0.5392, NMI: 0.3062, ASW: 0.1920, ARI: 0.2100, Delta: 0.0282
# Epoch 64 completed. Average Loss: 0.0657
# [Epoch 65] ACC: 0.5436, NMI: 0.3050, ASW: 0.1732, ARI: 0.2194, Delta: 0.0347
# Epoch 65 completed. Average Loss: 0.0719
# [Epoch 66] ACC: 0.5440, NMI: 0.3078, ASW: 0.1735, ARI: 0.2167, Delta: 0.0309
# Epoch 66 completed. Average Loss: 0.0742
# [Epoch 67] ACC: 0.5506, NMI: 0.3100, ASW: 0.1801, ARI: 0.2285, Delta: 0.0351
# Epoch 67 completed. Average Loss: 0.0788
# [Epoch 68] ACC: 0.5310, NMI: 0.3071, ASW: 0.1942, ARI: 0.1970, Delta: 0.0831
# Epoch 68 completed. Average Loss: 0.0751
# [Epoch 69] ACC: 0.5721, NMI: 0.3280, ASW: 0.1752, ARI: 0.2607, Delta: 0.1165
# Epoch 69 completed. Average Loss: 0.0719
# [Epoch 70] ACC: 0.5693, NMI: 0.3275, ASW: 0.1801, ARI: 0.2537, Delta: 0.0312
# Epoch 70 completed. Average Loss: 0.0888
# [Epoch 71] ACC: 0.5694, NMI: 0.3275, ASW: 0.1754, ARI: 0.2547, Delta: 0.0271
# Epoch 71 completed. Average Loss: 0.0851
# [Epoch 72] ACC: 0.5498, NMI: 0.3207, ASW: 0.1803, ARI: 0.2188, Delta: 0.0918
# Epoch 72 completed. Average Loss: 0.0878
# [Epoch 73] ACC: 0.5700, NMI: 0.3323, ASW: 0.1660, ARI: 0.2619, Delta: 0.0977
# Epoch 73 completed. Average Loss: 0.0981
# [Epoch 74] ACC: 0.5669, NMI: 0.3327, ASW: 0.1763, ARI: 0.2596, Delta: 0.0302
# Epoch 74 completed. Average Loss: 0.0971
# [Epoch 75] ACC: 0.5643, NMI: 0.3346, ASW: 0.1894, ARI: 0.2538, Delta: 0.0339
# Epoch 75 completed. Average Loss: 0.1046
# [Epoch 76] ACC: 0.5693, NMI: 0.3377, ASW: 0.1874, ARI: 0.2633, Delta: 0.0266
# Epoch 76 completed. Average Loss: 0.1034
# [Epoch 77] ACC: 0.5594, NMI: 0.3313, ASW: 0.1872, ARI: 0.2495, Delta: 0.0401
# Epoch 77 completed. Average Loss: 0.1141
# [Epoch 78] ACC: 0.5675, NMI: 0.3367, ASW: 0.1939, ARI: 0.2550, Delta: 0.0349
# Epoch 78 completed. Average Loss: 0.1169
# [Epoch 79] ACC: 0.5788, NMI: 0.3451, ASW: 0.2046, ARI: 0.2643, Delta: 0.0385
# Epoch 79 completed. Average Loss: 0.1278
# [Epoch 80] ACC: 0.5761, NMI: 0.3456, ASW: 0.2059, ARI: 0.2594, Delta: 0.0294
# Epoch 80 completed. Average Loss: 0.1377
# [Epoch 81] ACC: 0.5732, NMI: 0.3451, ASW: 0.2102, ARI: 0.2554, Delta: 0.0271
# Epoch 81 completed. Average Loss: 0.1337
# [Epoch 82] ACC: 0.5646, NMI: 0.3414, ASW: 0.2184, ARI: 0.2437, Delta: 0.0349
# Epoch 82 completed. Average Loss: 0.1375
# [Epoch 83] ACC: 0.5741, NMI: 0.3495, ASW: 0.2235, ARI: 0.2592, Delta: 0.0276
# Epoch 83 completed. Average Loss: 0.1346
# [Epoch 84] ACC: 0.5611, NMI: 0.3445, ASW: 0.2277, ARI: 0.2313, Delta: 0.0589
# Epoch 84 completed. Average Loss: 0.1413
# [Epoch 85] ACC: 0.5426, NMI: 0.3379, ASW: 0.2558, ARI: 0.2013, Delta: 0.0511
# Epoch 85 completed. Average Loss: 0.1384
# [Epoch 86] ACC: 0.5614, NMI: 0.3484, ASW: 0.2485, ARI: 0.2316, Delta: 0.0432
# Epoch 86 completed. Average Loss: 0.1365
# [Epoch 87] ACC: 0.5325, NMI: 0.3351, ASW: 0.2713, ARI: 0.1920, Delta: 0.0656
# Epoch 87 completed. Average Loss: 0.1516
# [Epoch 88] ACC: 0.5505, NMI: 0.3487, ASW: 0.2783, ARI: 0.2208, Delta: 0.0374
# Epoch 88 completed. Average Loss: 0.1475
# [Epoch 89] ACC: 0.5379, NMI: 0.3420, ASW: 0.2754, ARI: 0.2059, Delta: 0.0276
# Epoch 89 completed. Average Loss: 0.1625
# [Epoch 90] ACC: 0.5496, NMI: 0.3499, ASW: 0.2892, ARI: 0.2175, Delta: 0.0325
# Epoch 90 completed. Average Loss: 0.1538
# [Epoch 91] ACC: 0.5686, NMI: 0.3600, ASW: 0.2815, ARI: 0.2510, Delta: 0.0498
# Epoch 91 completed. Average Loss: 0.1581
# [Epoch 92] ACC: 0.5564, NMI: 0.3564, ASW: 0.2934, ARI: 0.2269, Delta: 0.0401
# Epoch 92 completed. Average Loss: 0.1586
# [Epoch 93] ACC: 0.5675, NMI: 0.3647, ASW: 0.2942, ARI: 0.2476, Delta: 0.0276
# Epoch 93 completed. Average Loss: 0.1681
# [Epoch 94] ACC: 0.5519, NMI: 0.3555, ASW: 0.2935, ARI: 0.2215, Delta: 0.0366
# Epoch 94 completed. Average Loss: 0.1566
# [Epoch 95] ACC: 0.5702, NMI: 0.3665, ASW: 0.3010, ARI: 0.2523, Delta: 0.0428
# Epoch 95 completed. Average Loss: 0.1760
# [Epoch 96] ACC: 0.5699, NMI: 0.3679, ASW: 0.3100, ARI: 0.2473, Delta: 0.0224
# Epoch 96 completed. Average Loss: 0.1899
# [Epoch 97] ACC: 0.5556, NMI: 0.3561, ASW: 0.3212, ARI: 0.2225, Delta: 0.0351
# Epoch 97 completed. Average Loss: 0.1747
# [Epoch 98] ACC: 0.5640, NMI: 0.3654, ASW: 0.3238, ARI: 0.2372, Delta: 0.0215
# Epoch 98 completed. Average Loss: 0.1797
# [Epoch 99] ACC: 0.5528, NMI: 0.3579, ASW: 0.3268, ARI: 0.2241, Delta: 0.0257
# Epoch 99 completed. Average Loss: 0.1836
# Final ACC: 0.5528